# 🪙 Gold Recovery Prediction — Full ML Pipeline
**Target**: `final.output.recovery` — the percentage of gold recovered in the flotation process.

**Pipeline Overview:**
1. Load & Inspect Data
2. Exploratory Data Analysis (EDA)
3. Data Cleaning & Imputation
4. Feature Engineering & Encoding
5. Model Training (XGBoost, LightGBM, Random Forest, Ridge, ElasticNet)
6. Model Comparison (R², RMSE, MAE)
7. Best Model Evaluation & Saving
8. Ideal Scenario Analysis (Maximising Gold Recovery)

---
## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
TARGET = 'final.output.recovery'

# Plotting style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#333',
    'text.color': '#e0e0e0',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'grid.color': '#2a2d3a',
    'grid.linestyle': '--',
    'font.family': 'monospace',
})

GOLD = '#FFD700'
SILVER = '#C0C0C0'
TEAL = '#00CED1'
CORAL = '#FF6B6B'

print('✅ Imports successful')

---
## 2. Load Dataset

In [ ]:
# ─── Load data ───────────────────────────────────────────────────────────────
# Replace 'gold_recovery.csv' with your actual file path
CSV_PATH = 'gold_recovery.csv'  # <-- update path if needed

df = pd.read_csv(CSV_PATH)

# Parse datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(f'Dataset shape : {df.shape}')
print(f'Date range    : {df["date"].min()}  →  {df["date"].max()}')
print(f'Target column : {TARGET}')
df.head(3)

---
## 3. Basic Dataset Overview

In [ ]:
print('=== DATA TYPES ===')
print(df.dtypes.value_counts())

print('\n=== MISSING VALUES (%) ===')
miss = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(miss[miss > 0].head(20))

print('\n=== DESCRIPTIVE STATISTICS ===')
df.describe().T

In [ ]:
# ─── Column categories ───────────────────────────────────────────────────────
non_feature_cols = ['date', TARGET]

# Identify leakage columns: other 'final.output.*' columns (they come after the target event)
leakage_cols = [c for c in df.columns if c.startswith('final.output.') and c != TARGET]
print('Leakage columns removed:', leakage_cols)

feature_cols = [c for c in df.columns if c not in non_feature_cols + leakage_cols]
print(f'\nTotal features : {len(feature_cols)}')

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ─── Target distribution ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Gold Recovery Rate — Target Distribution', color=GOLD, fontsize=14, fontweight='bold')

# Histogram
axes[0].hist(df[TARGET].dropna(), bins=50, color=GOLD, edgecolor='#0f1117', alpha=0.85)
axes[0].axvline(df[TARGET].mean(), color=CORAL, lw=2, label=f'Mean: {df[TARGET].mean():.2f}%')
axes[0].axvline(df[TARGET].median(), color=TEAL, lw=2, linestyle='--', label=f'Median: {df[TARGET].median():.2f}%')
axes[0].set_xlabel('Recovery Rate (%)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].set_title('Distribution')

# Box plot
bp = axes[1].boxplot(df[TARGET].dropna(), patch_artist=True,
                     boxprops=dict(facecolor=GOLD, alpha=0.6),
                     medianprops=dict(color=CORAL, lw=2),
                     whiskerprops=dict(color=SILVER),
                     capprops=dict(color=SILVER),
                     flierprops=dict(marker='o', color=TEAL, alpha=0.4, markersize=3))
axes[1].set_ylabel('Recovery Rate (%)')
axes[1].set_title('Box Plot')
axes[1].set_xticklabels(['Recovery'])

plt.tight_layout()
plt.savefig('plot_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Recovery over time ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
if df['date'].notna().any():
    ts = df.set_index('date')[TARGET].resample('D').mean()
    ax.plot(ts.index, ts.values, color=GOLD, lw=1.5, alpha=0.8)
    ax.fill_between(ts.index, ts.values, alpha=0.15, color=GOLD)
    ax.axhline(ts.mean(), color=CORAL, lw=1.5, linestyle='--', label=f'Overall Mean: {ts.mean():.2f}%')
    ax.set_xlabel('Date')
    ax.set_ylabel('Gold Recovery (%)')
    ax.set_title('Gold Recovery Rate Over Time (Daily Average)', color=GOLD, fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('plot_recovery_timeseries.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ─── Missing value heatmap ────────────────────────────────────────────────────
miss_pct = (df[feature_cols].isnull().sum() / len(df) * 100).sort_values(ascending=False)
miss_df = miss_pct[miss_pct > 0]

if len(miss_df) > 0:
    fig, ax = plt.subplots(figsize=(14, max(4, len(miss_df) * 0.35)))
    bars = ax.barh(miss_df.index, miss_df.values, color=CORAL, alpha=0.8, edgecolor='#0f1117')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Feature', color=CORAL, fontsize=13, fontweight='bold')
    ax.axvline(5, color=GOLD, lw=1, linestyle='--', label='5% threshold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('plot_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('✅ No missing values in feature columns!')

In [ ]:
# ─── Key feature correlations with target ─────────────────────────────────────
numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = df[numeric_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()

top_pos = corr_with_target.nlargest(15)
top_neg = corr_with_target.nsmallest(15)
top_corr = pd.concat([top_neg, top_pos])

fig, ax = plt.subplots(figsize=(12, 9))
colors = [CORAL if v < 0 else GOLD for v in top_corr.values]
ax.barh(top_corr.index, top_corr.values, color=colors, alpha=0.85, edgecolor='#0f1117')
ax.axvline(0, color=SILVER, lw=1)
ax.set_xlabel('Correlation with Recovery')
ax.set_title('Top Feature Correlations with Gold Recovery', color=GOLD, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 positive correlators:')
print(corr_with_target.nlargest(10))

In [ ]:
# ─── Correlation heatmap (key features) ──────────────────────────────────────
key_features = corr_with_target.abs().nlargest(15).index.tolist()
corr_matrix = df[key_features + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, cbar_kws={'shrink': 0.8},
            linewidths=0.5, linecolor='#0f1117',
            annot_kws={'size': 8})
ax.set_title('Correlation Heatmap — Top Features + Target', color=GOLD, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Key input distributions ─────────────────────────────────────────────────
input_features = [
    'rougher.input.feed_au',
    'rougher.input.feed_pb',
    'rougher.input.feed_ag',
    'rougher.input.feed_sol',
    'primary_cleaner.input.sulfate',
    'primary_cleaner.input.depressant',
]
input_features = [c for c in input_features if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
colors_cycle = [GOLD, SILVER, TEAL, CORAL, '#9B59B6', '#2ECC71']
for i, col in enumerate(input_features):
    axes[i].hist(df[col].dropna(), bins=40, color=colors_cycle[i], edgecolor='#0f1117', alpha=0.85)
    axes[i].set_title(col.split('.')[-1].replace('_', ' ').title(), fontsize=10)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
fig.suptitle('Key Input Feature Distributions', color=GOLD, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_input_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Data Cleaning & Feature Engineering

In [ ]:
# ─── Prepare working DataFrame ────────────────────────────────────────────────
df_model = df[feature_cols + [TARGET]].copy()

# Drop rows where target is missing
df_model.dropna(subset=[TARGET], inplace=True)
print(f'Rows after dropping missing target: {len(df_model)}')

# ─── Add time features from index if date available ───────────────────────────
if df['date'].notna().any():
    df_model['hour']       = df.loc[df_model.index, 'date'].dt.hour
    df_model['day_of_week'] = df.loc[df_model.index, 'date'].dt.dayofweek
    df_model['month']      = df.loc[df_model.index, 'date'].dt.month
    print('✅ Time features added: hour, day_of_week, month')

In [ ]:
# ─── Identify column types ────────────────────────────────────────────────────
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features : {len(numeric_features)}')
print(f'Categorical features: {len(cat_features)}')

# One-hot encode categorical features (if any)
if cat_features:
    X = pd.get_dummies(X, columns=cat_features, drop_first=True)
    print(f'Shape after OHE: {X.shape}')
    print('Encoded columns:', [c for c in X.columns if c not in numeric_features])
else:
    print('ℹ️  No categorical features — one-hot encoding not needed')

In [ ]:
# ─── Drop near-zero-variance columns ─────────────────────────────────────────
var = X.var()
low_var_cols = var[var < 1e-6].index.tolist()
print(f'Low-variance columns removed: {low_var_cols}')
X.drop(columns=low_var_cols, inplace=True)

# ─── KNN Imputation ────────────────────────────────────────────────────────────
print(f'Missing values before imputation: {X.isnull().sum().sum()}')
imputer = KNNImputer(n_neighbors=5)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
print(f'Missing values after imputation : {X_imputed.isnull().sum().sum()}')

# Save imputer
joblib.dump(imputer, 'imputer.pkl')
print('✅ Imputer saved to imputer.pkl')

In [ ]:
# ─── Train / Test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

# Save feature names for app.py
feature_names = X_imputed.columns.tolist()
joblib.dump(feature_names, 'feature_names.pkl')
print(f'Feature names saved ({len(feature_names)} features)')

---
## 6. Model Training & Comparison

In [ ]:
# ─── Define models ────────────────────────────────────────────────────────────
models = {
    'Ridge Regression': Ridge(alpha=10),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12,
                                            min_samples_leaf=5, n_jobs=-1,
                                            random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, max_depth=5,
                                                    learning_rate=0.05,
                                                    random_state=RANDOM_STATE),
    'XGBoost': xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=RANDOM_STATE, n_jobs=-1,
                                  verbosity=0),
    'LightGBM': lgb.LGBMRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8,
                                    random_state=RANDOM_STATE, n_jobs=-1,
                                    verbose=-1),
}

print('Models to train:', list(models.keys()))

In [ ]:
# ─── Train, evaluate, compare ─────────────────────────────────────────────────
results = []

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)

    results.append({'Model': name, 'R²': r2, 'RMSE': rmse, 'MAE': mae})
    print(f'  R²={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}')

results_df = pd.DataFrame(results).sort_values('R²', ascending=False).reset_index(drop=True)
print('\n=== MODEL COMPARISON ===')
print(results_df.to_string(index=False))

In [ ]:
# ─── Model comparison bar chart ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Performance Comparison', color=GOLD, fontsize=14, fontweight='bold')

metrics = ['R²', 'RMSE', 'MAE']
metric_colors = [GOLD, CORAL, TEAL]
model_names = results_df['Model'].tolist()

for i, (metric, color) in enumerate(zip(metrics, metric_colors)):
    vals = results_df[metric].tolist()
    bars = axes[i].barh(model_names, vals, color=color, alpha=0.85, edgecolor='#0f1117')
    axes[i].set_xlabel(metric)
    axes[i].set_title(metric, color=color, fontsize=12, fontweight='bold')
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                     f'{val:.4f}', va='center', fontsize=8, color='#e0e0e0')

plt.tight_layout()
plt.savefig('plot_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Best Model — Detailed Evaluation

In [ ]:
# ─── Select best model by R² ──────────────────────────────────────────────────
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
print(f'🏆 Best Model: {best_model_name}')
print(f'   R²   = {results_df.iloc[0]["R²"]:.4f}')
print(f'   RMSE = {results_df.iloc[0]["RMSE"]:.4f}')
print(f'   MAE  = {results_df.iloc[0]["MAE"]:.4f}')

In [ ]:
# ─── Cross-validation on best model ──────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(best_model, X_imputed, y, cv=kf, scoring='r2', n_jobs=-1)
print(f'5-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold   : {cv_scores.round(4)}')

In [ ]:
# ─── Actual vs Predicted ──────────────────────────────────────────────────────
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(f'{best_model_name} — Prediction Analysis', color=GOLD, fontsize=14, fontweight='bold')

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, alpha=0.4, color=GOLD, s=20, edgecolors='none')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], color=CORAL, lw=2, linestyle='--', label='Perfect Fit')
axes[0].set_xlabel('Actual Recovery (%)')
axes[0].set_ylabel('Predicted Recovery (%)')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

# Residuals
axes[1].hist(residuals, bins=50, color=TEAL, edgecolor='#0f1117', alpha=0.85)
axes[1].axvline(0, color=CORAL, lw=2, linestyle='--')
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residuals Distribution')

plt.tight_layout()
plt.savefig('plot_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Feature Importance ───────────────────────────────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X_imputed.columns)
    top20 = importances.nlargest(20)

    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(top20.index, top20.values, color=GOLD, alpha=0.85, edgecolor='#0f1117')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Top 20 Feature Importances — {best_model_name}',
                 color=GOLD, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plot_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Save top features for app
    top_features = top20.index.tolist()
    joblib.dump(top_features, 'top_features.pkl')
    print('Top features saved to top_features.pkl')
elif hasattr(best_model, 'coef_'):
    coefs = pd.Series(np.abs(best_model.coef_), index=X_imputed.columns)
    top20 = coefs.nlargest(20)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(top20.index, top20.values, color=GOLD, alpha=0.85)
    ax.set_title(f'Top 20 |Coefficients| — {best_model_name}',
                 color=GOLD, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('plot_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 8. Save Final Model (≤ 50 MB)

In [ ]:
# ─── Save model ───────────────────────────────────────────────────────────────
MODEL_PATH = 'gold_recovery_model.pkl'
joblib.dump(best_model, MODEL_PATH, compress=3)
size_mb = os.path.getsize(MODEL_PATH) / 1024**2
print(f'Model saved: {MODEL_PATH}  ({size_mb:.2f} MB)')

if size_mb > 50:
    print('⚠️  Model > 50 MB — reducing complexity...')
    if hasattr(best_model, 'n_estimators'):
        # Rebuild with fewer estimators
        params = best_model.get_params()
        params['n_estimators'] = max(50, params.get('n_estimators', 100) // 3)
        small_model = type(best_model)(**params)
        small_model.fit(X_train, y_train)
        joblib.dump(small_model, MODEL_PATH, compress=9)
        size_mb = os.path.getsize(MODEL_PATH) / 1024**2
        print(f'Reduced model saved: {size_mb:.2f} MB')
        best_model = small_model

assert size_mb <= 50, f'Model still > 50 MB ({size_mb:.2f} MB)!'
print(f'✅ Final model size: {size_mb:.2f} MB — within limit')

In [ ]:
# ─── Save metadata for Streamlit app ──────────────────────────────────────────
meta = {
    'best_model_name': best_model_name,
    'r2': float(results_df.iloc[0]['R²']),
    'rmse': float(results_df.iloc[0]['RMSE']),
    'mae': float(results_df.iloc[0]['MAE']),
    'n_features': len(feature_names),
    'n_train': len(X_train),
    'n_test': len(X_test),
    'n_total': len(df_model),
    'target': TARGET,
    'satisfactory_threshold': float(y.quantile(0.5)),  # median as threshold
    'high_recovery_threshold': float(y.quantile(0.75)),
    'target_mean': float(y.mean()),
    'target_std': float(y.std()),
    'target_min': float(y.min()),
    'target_max': float(y.max()),
    'results': results_df.to_dict(orient='records'),
    'feature_names': feature_names,
}
joblib.dump(meta, 'model_metadata.pkl')
print('✅ Metadata saved to model_metadata.pkl')

---
## 9. Ideal Scenario — Maximising Gold Recovery

In [ ]:
# ─── Find top-10% recovery rows & analyse ideal conditions ────────────────────
p90 = y.quantile(0.90)
ideal_mask = y >= p90
df_ideal = X_imputed[ideal_mask]
df_normal = X_imputed[~ideal_mask]

print(f'High recovery rows (≥{p90:.2f}%): {ideal_mask.sum()}')
print(f'Normal rows                      : {(~ideal_mask).sum()}')

In [ ]:
# ─── Compare mean values: ideal vs normal ─────────────────────────────────────
comparison = pd.DataFrame({
    'Ideal (Top 10%)': df_ideal.mean(),
    'Normal': df_normal.mean(),
})
comparison['Diff (Ideal - Normal)'] = comparison['Ideal (Top 10%)'] - comparison['Normal']
comparison['Diff %'] = (comparison['Diff (Ideal - Normal)'] / comparison['Normal'].abs() * 100).round(2)

# Top distinguishing features
top_diff = comparison['Diff %'].abs().nlargest(15)
print('=== Features Most Different in High-Recovery Scenarios ===')
print(comparison.loc[top_diff.index].to_string())

In [ ]:
# ─── Ideal scenario visualisation ─────────────────────────────────────────────
top15_diff_idx = top_diff.index
ideal_vals   = comparison.loc[top15_diff_idx, 'Ideal (Top 10%)']
normal_vals  = comparison.loc[top15_diff_idx, 'Normal']

short_names = [c.split('.')[-1][:25] for c in top15_diff_idx]
x = np.arange(len(top15_diff_idx))
width = 0.38

fig, ax = plt.subplots(figsize=(16, 7))
ax.bar(x - width/2, normal_vals,  width, label='Normal',        color=SILVER, alpha=0.75)
ax.bar(x + width/2, ideal_vals,   width, label='High Recovery',  color=GOLD,   alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Feature Value')
ax.set_title('Ideal vs Normal Conditions — Top Distinguishing Features',
             color=GOLD, fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('plot_ideal_scenario.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── Save ideal scenario profile ──────────────────────────────────────────────
ideal_profile = df_ideal.mean().to_dict()
joblib.dump(ideal_profile, 'ideal_profile.pkl')
print('✅ Ideal scenario profile saved to ideal_profile.pkl')

In [ ]:
# ─── Predict ideal scenario recovery ──────────────────────────────────────────
ideal_row = pd.DataFrame([ideal_profile])[feature_names].fillna(X_imputed.mean())
ideal_prediction = best_model.predict(ideal_row)[0]
print(f'🏅 Predicted recovery under IDEAL conditions: {ideal_prediction:.2f}%')
print(f'   (Actual top-10% mean: {y[ideal_mask].mean():.2f}%)')

---
## 10. Summary

In [ ]:
print('=' * 60)
print('   GOLD RECOVERY PREDICTION — FINAL SUMMARY')
print('=' * 60)
print(f'Dataset size     : {len(df_model)} rows, {len(feature_names)} features')
print(f'Target           : {TARGET}')
print(f'Target range     : {y.min():.2f}% — {y.max():.2f}%')
print(f'Target mean      : {y.mean():.2f}%')
print(f'Best model       : {best_model_name}')
print(f'  R²             : {results_df.iloc[0]["R²"]:.4f}')
print(f'  RMSE           : {results_df.iloc[0]["RMSE"]:.4f}')
print(f'  MAE            : {results_df.iloc[0]["MAE"]:.4f}')
print(f'Model size       : {os.path.getsize(MODEL_PATH)/1024**2:.2f} MB')
print(f'Ideal pred.      : {ideal_prediction:.2f}%')
print('=' * 60)
print('Files saved:')
for f in ['gold_recovery_model.pkl','imputer.pkl','feature_names.pkl',
           'model_metadata.pkl','ideal_profile.pkl','top_features.pkl']:
    if os.path.exists(f):
        print(f'  ✅ {f}  ({os.path.getsize(f)/1024:.1f} KB)')